<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/04_construction_review_indicators_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DADS5001 Mini Project
## ตัวชี้วัดและรูปแบบที่ควรตรวจต่อในโครงการจ้างก่อสร้างวงเงินไม่เกิน 500,000 บาท

ข้อมูล e-GP ปีงบประมาณ 2569 สะสมถึงวันที่ 30 กรกฎาคม 2569 ไม่ใช่ข้อมูลเต็มปี

Notebook นี้รับกลุ่มศึกษาหลักจาก EDA ได้แก่ **โครงการจ้างก่อสร้างวิธีเฉพาะเจาะจงที่มีวงเงินไม่เกิน 500,000 บาท** แล้วทดสอบที่มาของ threshold ก่อนสร้างตัวชี้วัดสามมิติ:

1. ส่วนต่างราคา
2. รูปแบบใกล้เพดานที่เกิดซ้ำ
3. การพึ่งพาผู้รับจ้างสูงภายในหน่วยงานย่อย

การผ่านตัวชี้วัดเป็นเหตุให้เปิดเอกสารตรวจต่อ ไม่ใช่ข้อสรุปความผิด


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ดาวน์โหลดฟอนต์ TH Sarabun New
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# เพิ่มฟอนต์ให้ Matplotlib
fm.fontManager.addfont(
    'thsarabunnew-webfont.ttf'
)

# กำหนดฟอนต์เริ่มต้นสำหรับ Matplotlib และ Seaborn
mpl.rc(
    'font',
    family='TH Sarabun New'
)
mpl.rcParams['axes.unicode_minus'] = False

sns.set_theme(
    style='whitegrid',
    font='TH Sarabun New'
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option(
    'display.float_format',
    lambda value: f'{value:,.2f}'
)

processed_dir = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/'
    'egp-contract/processed'
)

project_path = processed_dir / 'construction_projects_2569.csv'
contract_path = processed_dir / 'construction_contracts_2569.csv'
supplier_path = processed_dir / 'construction_supplier_summary_2569.csv'

In [ ]:
project_directory = processed_dir.parents[3]
figure_directory = project_directory / 'figure'
figure_directory.mkdir(parents=True, exist_ok=True)

project_data = pd.read_csv(
    project_path,
    low_memory=False
)
contract_data = pd.read_csv(
    contract_path,
    low_memory=False
)
supplier_data = pd.read_csv(
    supplier_path,
    low_memory=False
)

assert len(project_data) == 178_978
assert len(contract_data) == 180_079
assert len(supplier_data) == 33_570

print(f'Figure directory: {figure_directory}')
print(f'Project data: {project_data.shape}')
print(f'Contract data: {contract_data.shape}')
print(f'Supplier data: {supplier_data.shape}')
print('Input validation passed')


## 1. กรอบการวิเคราะห์

1. ตรวจ grain และผลรวมระดับโครงการ–สัญญา
2. จัดการแถวสมาชิก Joint Venture (JV)
3. ตรวจคุณภาพราคากลาง
4. ทดสอบ threshold หลายค่าก่อนเลือกเกณฑ์
5. สร้างตัวชี้วัดแต่ละมิติ
6. ตรวจการซ้อนทับข้ามมิติ
7. จัดลำดับโครงการและเลือกกรณีศึกษา

ทุก threshold ที่ทีมกำหนดต้องแยกจากเกณฑ์ตามกฎหมายอย่างชัดเจน


In [ ]:
# Column names used throughout this notebook
project_id_column = 'รหัสโครงการ'
project_name_column = 'ชื่อโครงการจัดซื้อจัดจ้าง'
agency_column = 'ชื่อหน่วยงาน'
province_column = 'จังหวัด'
method_column = 'ชื่อวิธีการจัดซื้อจัดจ้าง'
budget_column = 'วงเงินงบประมาณ (บาท)'
reference_price_column = 'ราคากลาง (บาท)'
awarded_price_column = 'ราคาที่ตกลงซื้อ / จ้าง ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
contract_number_column = 'เลขที่สัญญา'
contract_budget_column = 'วงเงินงบประมาณในสัญญา (บาท)'
supplier_id_column = 'เลขประจำตัวนิติบุคคล 13 หลัก'
supplier_name_column = 'ชื่อผู้ชนะการเสนอราคา'
subagency_column = 'ชื่อหน่วยงานย่อย'
announcement_date_column = 'วันที่ประกาศจัดซื้อจัดจ้าง'
transaction_date_column = 'วันที่เกิดรายการ'

In [ ]:
# Study scope and analytical thresholds
specific_method_value = 'เฉพาะเจาะจง'
legal_budget_ceiling = 500_000

near_ceiling_lower_bound = 490_000
minimum_cluster_size = 3

minimum_overrun_amount = 10_000
minimum_overrun_pct = 1

minimum_supplier_projects = 10
minimum_supplier_share = 75

project_data['in_study_scope'] = (
    project_data[method_column].eq(specific_method_value)
    & project_data[budget_column].le(legal_budget_ceiling)
)

study_project_ids = set(
    project_data.loc[
        project_data['in_study_scope'],
        project_id_column
    ]
)

study_population_summary = pd.Series({
    'โครงการก่อสร้างทั้งหมด': len(project_data),
    'โครงการในกลุ่มศึกษาหลัก': project_data['in_study_scope'].sum(),
    'สัดส่วนของโครงการก่อสร้างทั้งหมด (%)': (
        project_data['in_study_scope'].mean() * 100
    )
}, name='value')

display(study_population_summary)


## 2. ตัวชี้วัดคุณภาพข้อมูล

### 2.1 การตรวจยอดและโครงสร้าง Joint Venture (JV)

กลุ่มผู้รับจ้างแบบ Joint Venture (JV) คือผู้ประกอบการหลายราย
ร่วมกันรับงานภายใต้โครงการหรือสัญญาเดียวกัน

เริ่มจากเปรียบเทียบผลรวมวงเงินระดับสัญญากับราคาที่ตกลงระดับโครงการ
โดยยอมรับผลต่างจากการปัดเศษไม่เกิน 1 บาท ก่อนแยกโครงสร้าง JV


In [ ]:
contract_value_by_project = (
    contract_data
    .groupby(project_id_column)[contract_budget_column]
    .sum(min_count=1)
    .rename('contract_budget_sum')
    .reset_index()
)

project_data = (project_data.merge(contract_value_by_project, on=project_id_column, how='left'))

project_data['contract_value_difference'] = (project_data['contract_budget_sum'] - project_data[awarded_price_column])
project_data['contract_value_abs_difference'] = (project_data['contract_value_difference'].abs())
project_data['contract_value_difference_pct'] = (project_data['contract_value_difference'].div(project_data[awarded_price_column]).mul(100))
project_data['flag_contract_mismatch'] = ~np.isclose(project_data['contract_budget_sum'], project_data[awarded_price_column], rtol=0, atol=1, equal_nan=False)

In [ ]:
contract_mismatch_summary = pd.Series({
    'Total projects': (
        len(project_data)
    ),
    'Matched projects': (
        (~project_data[
            'flag_contract_mismatch'
        ]).sum()
    ),
    'Mismatch projects': (
        project_data[
            'flag_contract_mismatch'
        ].sum()
    ),
    'Mismatch pct': (
        project_data[
            'flag_contract_mismatch'
        ].mean()
        * 100
    ),
    'Total absolute difference': (
        project_data.loc[
            project_data[
                'flag_contract_mismatch'
            ],
            'contract_value_abs_difference'
        ].sum()
    )
})

display(
    contract_mismatch_summary
    .to_frame(name='value')
)

contract_mismatch_cases = (
    project_data
    .loc[
        project_data[
            'flag_contract_mismatch'
        ],
        [
            project_id_column,
            'ชื่อโครงการจัดซื้อจัดจ้าง',
            'ชื่อหน่วยงาน',
            method_column,
            awarded_price_column,
            'contract_budget_sum',
            'contract_value_difference',
            'contract_value_abs_difference',
            'contract_value_difference_pct'
        ]
    ]
    .sort_values(
        'contract_value_abs_difference',
        ascending=False
    )
)

display(
    contract_mismatch_cases.head(15)
)

In [ ]:
sample_mismatch_ids = (
    contract_mismatch_cases
    .head(5)[
        project_id_column
    ]
    .tolist()
)

sample_mismatch_rows = (
    contract_data
    .loc[
        contract_data[
            project_id_column
        ].isin(
            sample_mismatch_ids
        ),
        [
            project_id_column,
            'ชื่อโครงการจัดซื้อจัดจ้าง',
            supplier_id_column,
            supplier_name_column,
            'เลขที่สัญญา',
            contract_budget_column
        ]
    ]
    .sort_values(
        [
            project_id_column,
            'เลขที่สัญญา',
            supplier_id_column
        ]
    )
    .head(30)
)

display(sample_mismatch_rows)

In [ ]:
mismatch_project_profile = (
    contract_data
    .loc[
        contract_data[
            project_id_column
        ].isin(
            contract_mismatch_cases[
                project_id_column
            ]
        )
    ]
    .groupby(project_id_column)
    .agg(
        row_count=(
            project_id_column,
            'size'
        ),
        supplier_count=(
            supplier_id_column,
            'nunique'
        ),
        contract_number_count=(
            'เลขที่สัญญา',
            'nunique'
        ),
        contract_budget_value_count=(
            contract_budget_column,
            'nunique'
        )
    )
    .reset_index()
)

display(
    mismatch_project_profile
    .describe()
    .T
)

print(
    'Mismatch projects with multiple suppliers:',
    mismatch_project_profile[
        'supplier_count'
    ].gt(1).sum()
)

print(
    'Mismatch projects with multiple rows:',
    mismatch_project_profile[
        'row_count'
    ].gt(1).sum()
)

### การตีความส่วนต่างของมูลค่าระดับสัญญา

บางโครงการบันทึกทั้งแถวของกลุ่ม JV ซึ่งมีมูลค่าเต็มสัญญา
และแถวของบริษัทสมาชิกซึ่งมีมูลค่าตามส่วนแบ่ง หากรวมทุกแถว
จะทำให้มูลค่าถูกนับซ้ำ

ส่วนต่างดิบ 362 โครงการจึงยังไม่ถือเป็นปัญหาคุณภาพข้อมูลโดยอัตโนมัติ
การวิเคราะห์จะเก็บแถวของกลุ่ม JV และไม่นับแถวสมาชิกซ้ำ


In [ ]:
contract_data[
    'is_joint_venture_member'
] = (
    contract_data[
        supplier_name_column
    ]
    .astype('string')
    .str.contains(
        'สัญญากิจการค้าร่วม',
        na=False
    )
)

joint_venture_member_summary = pd.Series({
    'Joint-venture member rows': (
        contract_data[
            'is_joint_venture_member'
        ].sum()
    ),
    'Projects with member rows': (
        contract_data.loc[
            contract_data[
                'is_joint_venture_member'
            ],
            project_id_column
        ].nunique()
    ),
    'Member-row contract value': (
        contract_data.loc[
            contract_data[
                'is_joint_venture_member'
            ],
            contract_budget_column
        ].sum()
    )
})

display(
    joint_venture_member_summary
    .to_frame(name='value')
)

In [ ]:
supplier_entity_data = (
    contract_data
    .loc[
        ~contract_data[
            'is_joint_venture_member'
        ]
    ]
    .copy()
)

entity_value_by_project = (
    supplier_entity_data
    .groupby(project_id_column)
    [contract_budget_column]
    .sum(
        min_count=1
    )
    .rename('entity_contract_value_sum')
    .reset_index()
)

entity_reconciliation = (
    project_data
    [
        [
            project_id_column,
            awarded_price_column
        ]
    ]
    .merge(
        entity_value_by_project,
        on=project_id_column,
        how='left'
    )
)

entity_reconciliation[
    'difference'
] = (
    entity_reconciliation[
        'entity_contract_value_sum'
    ]
    - entity_reconciliation[
        awarded_price_column
    ]
)

entity_reconciliation[
    'is_matched'
] = np.isclose(
    entity_reconciliation[
        'entity_contract_value_sum'
    ],
    entity_reconciliation[
        awarded_price_column
    ],
    rtol=0,
    atol=1,
    equal_nan=False
)

entity_reconciliation_summary = pd.Series({
    'Supplier entity rows': (
        len(supplier_entity_data)
    ),
    'Projects checked': (
        len(entity_reconciliation)
    ),
    'Matched projects': (
        entity_reconciliation[
            'is_matched'
        ].sum()
    ),
    'Unmatched projects': (
        ~entity_reconciliation[
            'is_matched'
        ]
    ).sum(),
    'Matched pct': (
        entity_reconciliation[
            'is_matched'
        ].mean()
        * 100
    ),
    'Total absolute difference': (
        entity_reconciliation[
            'difference'
        ].abs().sum()
    )
})

display(
    entity_reconciliation_summary
    .to_frame(name='value')
)

### ผลการจัดการ Joint Venture (JV)

พบแถวสมาชิก JV 710 แถวใน 362 โครงการ มูลค่ารวม 13.06 พันล้านบาท หลังไม่นับแถวสมาชิกซ้ำ ยอดระดับสัญญาตรงกับระดับโครงการ 178,967 จาก 178,978 โครงการ หรือ 99.99%

โครงการที่ยังไม่ตรงกัน 11 โครงการถูกเก็บเป็นตัวชี้วัดคุณภาพข้อมูล ไม่รวมเป็นตัวชี้วัดเพื่อจัดลำดับการตรวจสอบโดยอัตโนมัติ


In [ ]:
multi_party_project_ids = set(
    contract_data.loc[
        contract_data[
            'is_joint_venture_member'
        ],
        project_id_column
    ]
)

entity_mismatch_project_ids = set(
    entity_reconciliation.loc[
        ~entity_reconciliation[
            'is_matched'
        ],
        project_id_column
    ]
)

project_data[
    'flag_multi_party_contract'
] = (
    project_data[
        project_id_column
    ].isin(
        multi_party_project_ids
    )
)

project_data[
    'flag_entity_value_mismatch'
] = (
    project_data[
        project_id_column
    ].isin(
        entity_mismatch_project_ids
    )
)

display(
    project_data[
        [
            'flag_multi_party_contract',
            'flag_entity_value_mismatch'
        ]
    ]
    .sum()
    .to_frame(name='project_count')
)

In [ ]:
required_supplier_columns = [
    supplier_id_column,
    'supplier_display_name',
    'supplier_project_count',
    'supplier_record_count',
    'total_contract_value',
    'median_contract_value',
    'total_contract_value_million',
    'value_share_pct'
]

missing_supplier_columns = [
    column
    for column in required_supplier_columns
    if column not in supplier_data.columns
]

assert not missing_supplier_columns, (
    f'Missing supplier columns: {missing_supplier_columns}'
)

supplier_summary = supplier_data.copy()

print(
    f'Supplier summary received from Notebook 03: '
    f'{supplier_summary.shape}'
)

In [ ]:
supplier_pareto = (
    supplier_summary
    .sort_values(
        'total_contract_value',
        ascending=False
    )
    .reset_index(drop=True)
    .copy()
)

supplier_pareto[
    'supplier_rank'
] = (
    supplier_pareto.index + 1
)

supplier_pareto[
    'cumulative_value_pct'
] = (
    supplier_pareto[
        'total_contract_value'
    ]
    .cumsum()
    .div(
        supplier_pareto[
            'total_contract_value'
        ].sum()
    )
    .mul(100)
)

total_suppliers = len(
    supplier_pareto
)

supplier_count_50 = (
    supplier_pareto[
        'cumulative_value_pct'
    ]
    .ge(50)
    .idxmax()
    + 1
)

supplier_count_80 = (
    supplier_pareto[
        'cumulative_value_pct'
    ]
    .ge(80)
    .idxmax()
    + 1
)

supplier_count_90 = (
    supplier_pareto[
        'cumulative_value_pct'
    ]
    .ge(90)
    .idxmax()
    + 1
)

corrected_concentration_summary = pd.Series({
    'Supplier entities': (
        total_suppliers
    ),
    'Top 1 value share (%)': (
        supplier_pareto
        .head(1)[
            'value_share_pct'
        ].sum()
    ),
    'Top 10 value share (%)': (
        supplier_pareto
        .head(10)[
            'value_share_pct'
        ].sum()
    ),
    'Top 100 value share (%)': (
        supplier_pareto
        .head(100)[
            'value_share_pct'
        ].sum()
    ),
    'Suppliers accounting for 50%': (
        supplier_count_50
    ),
    'Supplier pct accounting for 50%': (
        supplier_count_50
        / total_suppliers
        * 100
    ),
    'Suppliers accounting for 80%': (
        supplier_count_80
    ),
    'Supplier pct accounting for 80%': (
        supplier_count_80
        / total_suppliers
        * 100
    ),
    'Suppliers accounting for 90%': (
        supplier_count_90
    ),
    'Supplier pct accounting for 90%': (
        supplier_count_90
        / total_suppliers
        * 100
    )
})

display(
    supplier_pareto[
        [
            supplier_id_column,
            'supplier_display_name',
            'total_contract_value_million',
            'value_share_pct',
            'supplier_project_count'
        ]
    ]
    .head(15)
)

display(
    corrected_concentration_summary
    .to_frame(name='value')
)

### ผลตรวจข้อมูลผู้รับจ้างระดับประเทศ

หลังไม่นับแถวสมาชิก JV ซ้ำ พบผู้รับจ้างหรือกลุ่มผู้รับจ้าง 33,570 ราย โดย 1,870 ราย หรือ 5.57% ครองมูลค่าสะสม 80%

ผลระดับประเทศใช้เป็นบริบทเท่านั้น เพราะคำถามหลักต้องวิเคราะห์การพึ่งพาผู้รับจ้างภายในหน่วยงานย่อยและกลุ่มศึกษาหลัก


### 2.2 ราคากลางสูญหายหรือมีสัดส่วนผิดสังเกต

EDA พบว่าส่วนต่างระหว่างราคากลางกับราคาตกลงบางโครงการ
มีค่าร้อยละรุนแรง เพราะราคากลางมีค่าต่ำหรือสูงผิดสังเกต
เมื่อเทียบกับวงเงินงบประมาณ

ส่วนนี้ตรวจการกระจายของสัดส่วน
`ราคากลาง / วงเงินงบประมาณ` และใช้ผลดังกล่าว
จำแนกสถานะคุณภาพข้อมูลของราคากลาง

In [ ]:
project_data[
    'reference_to_budget_pct'
] = (
    project_data[
        reference_price_column
    ]
    .div(
        project_data[
            budget_column
        ]
    )
    .mul(100)
)

display(
    project_data[
        'reference_to_budget_pct'
    ]
    .describe(
        percentiles=[
            0.001,
            0.005,
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
    .to_frame(name='value')
)

In [ ]:
reference_ratio_summary = pd.Series({
    'Reference price missing': (
        project_data[
            reference_price_column
        ].isna().sum()
    ),
    'Reference below 1% of budget': (
        project_data[
            'reference_to_budget_pct'
        ].lt(1).sum()
    ),
    'Reference below 50% of budget': (
        project_data[
            'reference_to_budget_pct'
        ].lt(50).sum()
    ),
    'Reference above 150% of budget': (
        project_data[
            'reference_to_budget_pct'
        ].gt(150).sum()
    ),
    'Reference above 200% of budget': (
        project_data[
            'reference_to_budget_pct'
        ].gt(200).sum()
    )
})

display(
    reference_ratio_summary
    .to_frame(name='project_count')
)

In [ ]:
# Inspect projects with unusually low reference prices
reference_case_columns = [
    project_id_column,
    project_name_column,
    agency_column,
    method_column,
    budget_column,
    reference_price_column,
    awarded_price_column,
    'reference_to_budget_pct',
    'reference_discount_pct'
]

low_reference_cases = (
    project_data.loc[
        project_data['reference_to_budget_pct'] < 50,
        reference_case_columns
    ]
    .sort_values('reference_to_budget_pct')
    .head(10)
)

print('Projects with the lowest reference-price-to-budget ratios:')
display(
    low_reference_cases.style.format({
        budget_column: '{:,.2f}',
        reference_price_column: '{:,.2f}',
        awarded_price_column: '{:,.2f}',
        'reference_to_budget_pct': '{:,.2f}',
        'reference_discount_pct': '{:,.2f}'
    })
)

In [ ]:
# Inspect projects with unusually high reference prices
high_reference_cases = (
    project_data.loc[
        project_data['reference_to_budget_pct'] > 150,
        reference_case_columns
    ]
    .sort_values('reference_to_budget_pct', ascending=False)
    .head(10)
)

print('Projects with the highest reference-price-to-budget ratios:')
display(
    high_reference_cases.style.format({
        budget_column: '{:,.2f}',
        reference_price_column: '{:,.2f}',
        awarded_price_column: '{:,.2f}',
        'reference_to_budget_pct': '{:,.2f}',
        'reference_discount_pct': '{:,.2f}'
    })
)

### การตีความคุณภาพราคากลาง

ราคากลางต่องบประมาณมีค่ามัธยฐาน 100% แต่มีค่าปลายหางรุนแรง พบราคากลางต่ำกว่า 50% ของงบประมาณ 426 โครงการ สูงกว่า 150% จำนวน 280 โครงการ และขาดข้อมูล 105 โครงการ

จึงใช้เฉพาะสถานะ `Within expected range` สำหรับเปรียบเทียบราคาที่ตกลงกับราคากลาง ส่วนสถานะอื่นเป็นประเด็นคุณภาพข้อมูล ไม่ใช้ตัดสินโครงการ


In [ ]:
# Classify reference-price data quality
reference_ratio = project_data['reference_to_budget_pct']

project_data['flag_reference_price_missing'] = (
    project_data[reference_price_column].isna()
)

project_data['flag_reference_price_unusual'] = (
    project_data[reference_price_column].notna()
    & (
        (reference_ratio < 50)
        | (reference_ratio > 150)
    )
)

project_data['flag_reference_price_extreme'] = (
    project_data[reference_price_column].notna()
    & (
        (reference_ratio < 1)
        | (reference_ratio > 200)
    )
)

project_data['reference_price_status'] = np.select(
    [
        project_data['flag_reference_price_missing'],
        project_data['flag_reference_price_extreme'],
        project_data['flag_reference_price_unusual']
    ],
    [
        'Missing',
        'Extreme',
        'Unusual'
    ],
    default='Within expected range'
)

reference_status_summary = (
    project_data['reference_price_status']
    .value_counts()
    .rename_axis('reference_price_status')
    .reset_index(name='project_count')
)

reference_status_summary['project_pct'] = (
    reference_status_summary['project_count']
    / len(project_data)
    * 100
)

display(reference_status_summary)

## 3. ตัวชี้วัดเพื่อจัดลำดับการตรวจสอบ

### 3.1 รูปแบบใกล้เพดานที่เกิดซ้ำ

500,000 บาทเป็นเส้นแบ่งตามกฎหมาย แต่ช่วง 490,000–500,000 บาทและจำนวนขั้นต่ำของกลุ่มเป็นเกณฑ์วิเคราะห์ของทีม จึงทดสอบสองมิติ:

- หน้าต่างวงเงิน 450–500K, 480–500K, 490–500K และ 495–500K
- จำนวนขั้นต่ำของกลุ่ม 2, 3, 5 และ 10 โครงการ

กลุ่มเกิดซ้ำกำหนดจากหน่วยงานย่อย ผู้รับจ้าง และวันที่เกิดรายการเดียวกัน เนื่องจากโครงการเฉพาะเจาะจงไม่มีวันที่ประกาศที่ใช้งานได้


In [ ]:
project_data['flag_specific_near_500k'] = (
    project_data['in_study_scope']
    & project_data[budget_column].between(
        near_ceiling_lower_bound,
        legal_budget_ceiling,
        inclusive='both'
    )
)

study_projects = project_data.loc[
    project_data['in_study_scope']
]

near_500k_check = project_data.loc[
    project_data['flag_specific_near_500k']
]

specific_near_500k_summary = pd.DataFrame({
    'value': [
        len(study_projects),
        len(near_500k_check),
        len(near_500k_check) / len(study_projects) * 100
    ]
}, index=[
    'โครงการในกลุ่มศึกษาหลัก',
    'โครงการช่วง 490,000–500,000 บาท',
    'สัดส่วนในกลุ่มศึกษาหลัก (%)'
])

display(specific_near_500k_summary)


In [ ]:
near_ceiling_windows = [
    (450_000, '450,000–500,000'),
    (480_000, '480,000–500,000'),
    (490_000, '490,000–500,000'),
    (495_000, '495,000–500,000')
]

near_ceiling_sensitivity_records = []

for lower_bound, window_label in near_ceiling_windows:
    window_data = project_data.loc[
        project_data['in_study_scope']
        & project_data[budget_column].between(
            lower_bound,
            legal_budget_ceiling,
            inclusive='both'
        )
    ]

    window_clusters = (
        window_data
        .groupby(
            [
                subagency_column,
                supplier_id_column,
                transaction_date_column
            ],
            dropna=False
        )[project_id_column]
        .nunique()
    )

    near_ceiling_sensitivity_records.append({
        'ช่วงวงเงิน': window_label,
        'โครงการในช่วง': len(window_data),
        'cluster อย่างน้อย 3 โครงการ': (
            window_clusters.ge(minimum_cluster_size).sum()
        ),
        'โครงการใน cluster': (
            window_clusters.loc[
                window_clusters.ge(minimum_cluster_size)
            ].sum()
        )
    })

near_ceiling_sensitivity = pd.DataFrame(
    near_ceiling_sensitivity_records
)

display(near_ceiling_sensitivity)


In [ ]:
# Check available dates among specific-method projects near 500K
near_500k_check = project_data.loc[
    project_data['flag_specific_near_500k']
]

date_availability_summary = pd.DataFrame({
    'date_column': [
        announcement_date_column,
        transaction_date_column
    ],
    'usable_project_count': [
        (
            near_500k_check[announcement_date_column]
            .notna()
            & near_500k_check[announcement_date_column].ne('-')
        ).sum(),
        (
            near_500k_check[transaction_date_column]
            .notna()
            & near_500k_check[transaction_date_column].ne('-')
        ).sum()
    ],
    'missing_or_dash_count': [
        (
            near_500k_check[announcement_date_column].isna()
            | near_500k_check[announcement_date_column].eq('-')
        ).sum(),
        (
            near_500k_check[transaction_date_column].isna()
            | near_500k_check[transaction_date_column].eq('-')
        ).sum()
    ]
})

display(date_availability_summary)

print('Most frequent announcement dates:')
display(
    near_500k_check[announcement_date_column]
    .value_counts(dropna=False)
    .head(10)
)

print('Most frequent transaction dates:')
display(
    near_500k_check[transaction_date_column]
    .value_counts(dropna=False)
    .head(10)
)

In [ ]:
# Identify repeated near-500K projects recorded on the same date
cluster_columns = [
    subagency_column,
    supplier_id_column,
    transaction_date_column
]

near_500k_clusters = (
    near_500k_check
    .groupby(cluster_columns, dropna=False)
    .agg(
        project_count=(project_id_column, 'nunique'),
        total_budget=(budget_column, 'sum'),
        total_awarded_value=(awarded_price_column, 'sum'),
        supplier_name=(supplier_name_column, 'first')
    )
    .reset_index()
)

repeated_near_500k_clusters = (
    near_500k_clusters.loc[
        near_500k_clusters['project_count'] >= 2
    ]
    .sort_values(
        ['project_count', 'total_budget'],
        ascending=False
    )
)

cluster_screening_summary = pd.DataFrame({
    'value': [
        len(near_500k_check),
        len(repeated_near_500k_clusters),
        repeated_near_500k_clusters['project_count'].sum(),
        (repeated_near_500k_clusters['project_count'] >= 3).sum(),
        repeated_near_500k_clusters.loc[
            repeated_near_500k_clusters['project_count'] >= 3,
            'project_count'
        ].sum()
    ]
}, index=[
    'Specific near-500K projects',
    'Clusters with at least 2 projects',
    'Projects in clusters with at least 2 projects',
    'Clusters with at least 3 projects',
    'Projects in clusters with at least 3 projects'
])

display(cluster_screening_summary)

display(
    repeated_near_500k_clusters.head(20).style.format({
        'project_count': '{:,.0f}',
        'total_budget': '{:,.2f}',
        'total_awarded_value': '{:,.2f}'
    })
)

In [ ]:
# Create a repeated near-500K pattern indicator
minimum_cluster_size = 3

cluster_size_lookup = (
    near_500k_clusters[
        cluster_columns + ['project_count']
    ]
    .rename(columns={
        'project_count': 'near_500k_cluster_size'
    })
)

project_data = (
    project_data
    .drop(
        columns=['near_500k_cluster_size'],
        errors='ignore'
    )
    .merge(
        cluster_size_lookup,
        on=cluster_columns,
        how='left',
        validate='many_to_one'
    )
)

project_data['near_500k_cluster_size'] = (
    project_data['near_500k_cluster_size']
    .fillna(0)
    .astype(int)
)

project_data['flag_repeated_near_500k_pattern'] = (
    project_data['flag_specific_near_500k']
    & (
        project_data['near_500k_cluster_size']
        >= minimum_cluster_size
    )
)

repeated_pattern_count = (
    project_data['flag_repeated_near_500k_pattern'].sum()
)

repeated_pattern_summary = pd.DataFrame({
    'value': [
        repeated_pattern_count,
        repeated_pattern_count / len(project_data) * 100,
        repeated_pattern_count / len(near_500k_check) * 100,
        (
            project_data.loc[
                project_data['flag_repeated_near_500k_pattern'],
                budget_column
            ].sum()
            / 1_000_000_000
        )
    ]
}, index=[
    'Projects in repeated near-500K patterns',
    'Share of all construction projects (%)',
    'Share of specific near-500K projects (%)',
    'Total budget of flagged projects (billion THB)'
])

display(repeated_pattern_summary)

In [ ]:
cluster_size_options = [2, 3, 5, 10]

cluster_size_sensitivity = pd.DataFrame([
    {
        'minimum_cluster_size': size,
        'cluster_count': (
            near_500k_clusters['project_count'].ge(size).sum()
        ),
        'project_count': (
            near_500k_clusters.loc[
                near_500k_clusters['project_count'].ge(size),
                'project_count'
            ].sum()
        )
    }
    for size in cluster_size_options
])

near_ceiling_sensitivity['repeated_project_pct'] = (
    near_ceiling_sensitivity['โครงการใน cluster']
    / near_ceiling_sensitivity['โครงการในช่วง']
    * 100
)

display(cluster_size_sensitivity)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].barh(
    near_ceiling_sensitivity['ช่วงวงเงิน'],
    near_ceiling_sensitivity['repeated_project_pct'],
    color='#4C78A8'
)
axes[0].bar_label(
    bars,
    labels=[
        (
            f'{percentage:.1f}% '
            f'({count:,.0f} โครงการ)'
        )
        for percentage, count in zip(
            near_ceiling_sensitivity['repeated_project_pct'],
            near_ceiling_sensitivity['โครงการใน cluster']
        )
    ],
    padding=4
)
axes[0].set_title('ผลเมื่อเปลี่ยนหน้าต่างใกล้เพดาน')
axes[0].set_xlabel('โครงการในกลุ่มเกิดซ้ำ (%)')
axes[0].set_ylabel('ช่วงวงเงิน')
axes[0].set_xlim(
    0,
    near_ceiling_sensitivity[
        'repeated_project_pct'
    ].max() * 1.38
)
axes[0].spines[['top', 'right', 'left']].set_visible(False)

bars = axes[1].bar(
    cluster_size_sensitivity['minimum_cluster_size'].astype(str),
    cluster_size_sensitivity['project_count'],
    color=[
        '#8FA3B8',
        '#E67E22',
        '#8FA3B8',
        '#8FA3B8'
    ]
)
axes[1].bar_label(
    bars,
    labels=[
        (
            f'{projects:,.0f} โครงการ\n'
            f'{clusters:,.0f} กลุ่ม'
        )
        for projects, clusters in zip(
            cluster_size_sensitivity['project_count'],
            cluster_size_sensitivity['cluster_count']
        )
    ],
    padding=4
)
axes[1].set_title('ผลเมื่อเปลี่ยนจำนวนขั้นต่ำของกลุ่ม')
axes[1].set_xlabel('จำนวนโครงการขั้นต่ำในกลุ่ม')
axes[1].set_ylabel('โครงการที่อยู่ในกลุ่มเกิดซ้ำ')
axes[1].set_ylim(
    0,
    cluster_size_sensitivity['project_count'].max() * 1.20
)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()

figure_path = (
    figure_directory
    / 'fig04_01_near_ceiling_sensitivity.png'
)
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'Figure saved: {figure_path}')


In [ ]:
# Inspect projects in the largest repeated near-500K cluster
largest_cluster = repeated_near_500k_clusters.iloc[0]

largest_cluster_projects = project_data.loc[
    project_data[subagency_column].eq(
        largest_cluster[subagency_column]
    )
    & project_data[supplier_id_column].eq(
        largest_cluster[supplier_id_column]
    )
    & project_data[transaction_date_column].eq(
        largest_cluster[transaction_date_column]
    )
    & project_data['flag_repeated_near_500k_pattern']
]

largest_cluster_columns = [
    project_id_column,
    project_name_column,
    province_column,
    budget_column,
    awarded_price_column,
    supplier_name_column,
    transaction_date_column
]

print(
    'Largest cluster:',
    largest_cluster[subagency_column],
    '|',
    largest_cluster['supplier_name'],
    '|',
    largest_cluster[transaction_date_column]
)

display(
    largest_cluster_projects[
        largest_cluster_columns
    ]
    .sort_values(
        budget_column,
        ascending=False
    )
    .head(15)
    .style.format({
        budget_column: '{:,.2f}',
        awarded_price_column: '{:,.2f}'
    })
)

### เหตุผลที่เลือกหน้าต่างและจำนวนขั้นต่ำ

หน้าต่าง 490,000–500,000 บาทแคบพอที่จะเน้นบริเวณก่อนถึงเพดานและยังมีข้อมูลเพียงพอ การเปลี่ยนหน้าต่างใช้ตรวจว่ารูปแบบไม่ได้ขึ้นกับขอบล่างค่าเดียว

จำนวนขั้นต่ำ 3 โครงการเป็นจุดที่เริ่มมองเป็นรูปแบบเกิดซ้ำมากกว่าคู่เหตุการณ์ ขณะที่การทดลอง 2, 5 และ 10 โครงการแสดง trade-off ระหว่างความไวกับจำนวนกรณีที่เหลือให้ตรวจ

ทั้งสองค่าเป็นเกณฑ์ของการศึกษา ไม่ใช่ข้อกำหนดทางกฎหมาย


### 3.2 ส่วนต่างราคา

การสำรวจพบโครงการที่ราคาตกลงสูงกว่างบประมาณหรือราคากลาง แต่บางกรณีมีส่วนต่างเพียงเล็กน้อย จึงต้องทดลองทั้งเกณฑ์จำนวนเงินและร้อยละก่อนเลือกค่าใช้งาน

ค่าที่ทดลอง:

- จำนวนเงิน 5,000 / 10,000 / 25,000 / 50,000 บาท
- สัดส่วน 0.5% / 1% / 5%

เกณฑ์เหล่านี้เป็น operational threshold ของการศึกษา ไม่ใช่ materiality ตามกฎหมายหรือมาตรฐานการตรวจสอบ


In [ ]:
price_tolerance = 1

project_data['budget_overrun'] = (
    project_data[awarded_price_column]
    - project_data[budget_column]
)

project_data['reference_overrun'] = (
    project_data[awarded_price_column]
    - project_data[reference_price_column]
)

project_data['flag_awarded_above_budget'] = (
    project_data['in_study_scope']
    & project_data['budget_overrun'].gt(price_tolerance)
)

project_data['flag_awarded_above_reference'] = (
    project_data['in_study_scope']
    & project_data['reference_price_status'].eq('Within expected range')
    & project_data['reference_overrun'].gt(price_tolerance)
)

study_valid_reference_count = (
    project_data['in_study_scope']
    & project_data['reference_price_status'].eq('Within expected range')
).sum()

price_flag_summary = pd.DataFrame({
    'indicator': [
        'ราคาตกลงสูงกว่างบประมาณ',
        'ราคาตกลงสูงกว่าราคากลางที่ใช้งานได้'
    ],
    'project_count': [
        project_data['flag_awarded_above_budget'].sum(),
        project_data['flag_awarded_above_reference'].sum()
    ],
    'eligible_projects': [
        project_data['in_study_scope'].sum(),
        study_valid_reference_count
    ]
})

price_flag_summary['project_pct'] = (
    price_flag_summary['project_count']
    / price_flag_summary['eligible_projects']
    * 100
)

display(price_flag_summary.round({'project_pct': 2}))


In [ ]:
project_data['budget_overrun_pct'] = (
    project_data['budget_overrun']
    / project_data[budget_column]
    * 100
)

project_data['reference_overrun_pct'] = (
    project_data['reference_overrun']
    / project_data[reference_price_column]
    * 100
)

above_budget_data = project_data.loc[
    project_data['flag_awarded_above_budget']
]

above_reference_data = project_data.loc[
    project_data['flag_awarded_above_reference']
]

price_overrun_summary = pd.DataFrame({
    'indicator': [
        'สูงกว่างบประมาณ',
        'สูงกว่าราคากลางที่ใช้งานได้'
    ],
    'project_count': [
        len(above_budget_data),
        len(above_reference_data)
    ],
    'total_overrun_million': [
        above_budget_data['budget_overrun'].sum() / 1_000_000,
        above_reference_data['reference_overrun'].sum() / 1_000_000
    ],
    'median_overrun': [
        above_budget_data['budget_overrun'].median(),
        above_reference_data['reference_overrun'].median()
    ],
    'median_overrun_pct': [
        above_budget_data['budget_overrun_pct'].median(),
        above_reference_data['reference_overrun_pct'].median()
    ]
})

display(price_overrun_summary)


In [ ]:
overrun_amount_thresholds = [5_000, 10_000, 25_000, 50_000]
overrun_pct_thresholds = [0.5, 1, 5]

materiality_records = []

for comparison, difference_column, pct_column, eligible_mask in [
    (
        'สูงกว่างบประมาณ',
        'budget_overrun',
        'budget_overrun_pct',
        project_data['in_study_scope']
    ),
    (
        'สูงกว่าราคากลาง',
        'reference_overrun',
        'reference_overrun_pct',
        (
            project_data['in_study_scope']
            & project_data['reference_price_status']
            .eq('Within expected range')
        )
    )
]:
    for amount_threshold in overrun_amount_thresholds:
        for pct_threshold in overrun_pct_thresholds:
            flag = (
                eligible_mask
                & project_data[difference_column].gt(amount_threshold)
                & project_data[pct_column].gt(pct_threshold)
            )
            materiality_records.append({
                'comparison': comparison,
                'amount_threshold': amount_threshold,
                'pct_threshold': pct_threshold,
                'project_count': flag.sum()
            })

materiality_sensitivity = pd.DataFrame(materiality_records)

display(
    materiality_sensitivity.pivot_table(
        index=['amount_threshold', 'pct_threshold'],
        columns='comparison',
        values='project_count'
    )
)


In [ ]:
project_data['flag_material_above_budget'] = (
    project_data['in_study_scope']
    & project_data['budget_overrun'].gt(minimum_overrun_amount)
    & project_data['budget_overrun_pct'].gt(minimum_overrun_pct)
)

project_data['flag_material_above_reference'] = (
    project_data['in_study_scope']
    & project_data['reference_price_status'].eq('Within expected range')
    & project_data['reference_overrun'].gt(minimum_overrun_amount)
    & project_data['reference_overrun_pct'].gt(minimum_overrun_pct)
)

project_data['price_review_status'] = np.select(
    [
        (
            project_data['flag_material_above_budget']
            & project_data['flag_material_above_reference']
        ),
        project_data['flag_material_above_budget'],
        project_data['flag_material_above_reference']
    ],
    [
        'สูงกว่าทั้งงบประมาณและราคากลาง',
        'สูงกว่างบประมาณอย่างเดียว',
        'สูงกว่าราคากลางอย่างเดียว'
    ],
    default='ไม่เข้าเกณฑ์'
)

price_review_summary = (
    project_data.loc[project_data['in_study_scope']]
    ['price_review_status']
    .value_counts()
    .rename_axis('price_review_status')
    .reset_index(name='project_count')
)

price_review_summary['project_pct'] = (
    price_review_summary['project_count']
    / project_data['in_study_scope'].sum()
    * 100
)

display(price_review_summary)


In [ ]:
price_heatmaps = {}

for comparison in [
    'สูงกว่างบประมาณ',
    'สูงกว่าราคากลาง'
]:
    price_heatmaps[comparison] = (
        materiality_sensitivity.loc[
            materiality_sensitivity['comparison'].eq(comparison)
        ]
        .pivot(
            index='pct_threshold',
            columns='amount_threshold',
            values='project_count'
        )
        .reindex(
            index=overrun_pct_thresholds,
            columns=overrun_amount_thresholds
        )
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, comparison in zip(
    axes,
    ['สูงกว่างบประมาณ', 'สูงกว่าราคากลาง']
):
    sns.heatmap(
        price_heatmaps[comparison],
        annot=True,
        fmt=',.0f',
        cmap='Blues',
        cbar=False,
        ax=ax
    )
    ax.add_patch(
        plt.Rectangle(
            (1, 1),
            1,
            1,
            fill=False,
            edgecolor='#D62728',
            linewidth=3
        )
    )
    ax.set_title(f'ราคาตกลง{comparison}')
    ax.set_xlabel('ส่วนต่างขั้นต่ำ (บาท)')
    ax.set_ylabel('ส่วนต่างขั้นต่ำ (%)')
    ax.set_xticklabels(
        [f'{value:,.0f}' for value in overrun_amount_thresholds]
    )

fig.suptitle(
    'จำนวนโครงการเปลี่ยนตามเกณฑ์ส่วนต่างราคา',
    fontsize=16
)

plt.tight_layout()

figure_path = (
    figure_directory
    / 'fig04_02_price_threshold_sensitivity.png'
)
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'Figure saved: {figure_path}')


In [ ]:
selected_price_summary = (
    price_review_summary.loc[
        price_review_summary['price_review_status']
        .ne('ไม่เข้าเกณฑ์')
    ]
    .sort_values('project_count', ascending=True)
    .copy()
)

fig, ax = plt.subplots(figsize=(10, 4.8))

bars = ax.barh(
    selected_price_summary['price_review_status'],
    selected_price_summary['project_count'],
    color=['#D98E73', '#D96C4A', '#B43C2F']
)

ax.bar_label(
    bars,
    labels=[
        f'{count:,.0f} โครงการ'
        for count in selected_price_summary['project_count']
    ],
    padding=4
)

ax.set_title(
    'โครงการที่ผ่านเกณฑ์ส่วนต่างมากกว่า 10,000 บาทและมากกว่า 1%'
)
ax.set_xlabel('จำนวนโครงการ')
ax.set_ylabel('')
ax.set_xlim(
    0,
    selected_price_summary['project_count'].max() * 1.22
)
ax.spines[['top', 'right', 'left']].set_visible(False)

plt.tight_layout()

figure_path = (
    figure_directory
    / 'fig04_03_selected_price_review_indicators.png'
)
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'Figure saved: {figure_path}')


In [ ]:
# Inspect projects with the largest material price overruns
price_review_cases = project_data.loc[
    project_data['price_review_status'].ne('ไม่เข้าเกณฑ์')
].copy()

price_review_cases['largest_overrun'] = (
    price_review_cases[
        ['budget_overrun', 'reference_overrun']
    ]
    .clip(lower=0)
    .max(axis=1)
)

price_review_columns = [
    project_id_column,
    project_name_column,
    agency_column,
    method_column,
    budget_column,
    reference_price_column,
    awarded_price_column,
    'budget_overrun',
    'budget_overrun_pct',
    'reference_overrun',
    'reference_overrun_pct',
    'price_review_status'
]

largest_price_review_cases = (
    price_review_cases
    .sort_values('largest_overrun', ascending=False)
    [price_review_columns]
    .head(20)
)

display(
    largest_price_review_cases.style.format({
        budget_column: '{:,.2f}',
        reference_price_column: '{:,.2f}',
        awarded_price_column: '{:,.2f}',
        'budget_overrun': '{:,.2f}',
        'budget_overrun_pct': '{:,.2f}',
        'reference_overrun': '{:,.2f}',
        'reference_overrun_pct': '{:,.2f}'
    })
)

### เหตุผลที่เลือกเกณฑ์ราคา

Sensitivity analysis แสดงว่าจำนวนเงินขั้นต่ำมีผลต่อจำนวนโครงการมากกว่าการขยับร้อยละในช่วงที่ทดลอง เกณฑ์ 5,000 บาทยังให้กรณีจำนวนมาก ขณะที่ 25,000–50,000 บาทตัดกรณีออกมาก จึงเลือกมากกว่า 10,000 บาทร่วมกับมากกว่า 1%

เกณฑ์จำนวนเงินช่วยกรองส่วนต่างเล็ก ส่วนเกณฑ์ร้อยละช่วยเทียบกับขนาดโครงการ ทั้งสองเงื่อนไขต้องผ่านพร้อมกัน ผลจริงหลัง Run all จะใช้เขียนจำนวนโครงการใน README


### 3.3 ความสัมพันธ์ที่หน่วยงานย่อยพึ่งพาผู้รับจ้างรายเดียวสูง

วิเคราะห์ภายในกลุ่มศึกษาหลัก เพราะบริบทและวิธีจัดซื้อควรสอดคล้องกัน ข้อมูลมีเฉพาะผู้ชนะ จึงไม่สามารถวัดการแข่งขันของผู้เสนอราคาทั้งหมดได้

เกณฑ์ขั้นต่ำ 10 โครงการและส่วนแบ่ง 75% เป็นสมมติฐานของทีม ไม่ใช่กฎหมายหรือมาตรฐานสากล จึงต้องแสดง sensitivity ก่อนเลือกเกณฑ์


In [ ]:
study_supplier_entity_data = (
    supplier_entity_data.loc[
        supplier_entity_data[project_id_column]
        .isin(study_project_ids)
    ]
    .copy()
)

supplier_relationships = (
    study_supplier_entity_data
    .groupby(
        [subagency_column, supplier_id_column],
        dropna=False
    )
    .agg(
        supplier_name=(supplier_name_column, 'first'),
        supplier_project_count=(project_id_column, 'nunique'),
        total_contract_value=(contract_budget_column, 'sum')
    )
    .reset_index()
)

subagency_totals = (
    study_supplier_entity_data
    .groupby(subagency_column, dropna=False)
    .agg(
        subagency_project_count=(project_id_column, 'nunique'),
        subagency_contract_value=(contract_budget_column, 'sum')
    )
    .reset_index()
)

supplier_relationships = supplier_relationships.merge(
    subagency_totals,
    on=subagency_column,
    how='left',
    validate='many_to_one'
)

supplier_relationships['project_share_within_subagency_pct'] = (
    supplier_relationships['supplier_project_count']
    / supplier_relationships['subagency_project_count']
    * 100
)

supplier_relationships['value_share_within_subagency_pct'] = (
    supplier_relationships['total_contract_value']
    / supplier_relationships['subagency_contract_value']
    * 100
)

display(
    supplier_relationships
    .sort_values('supplier_project_count', ascending=False)
    .head(20)
)


In [ ]:
minimum_project_options = [5, 10, 20]
share_threshold_options = [50, 75, 90]

concentration_sensitivity_records = []

for minimum_projects in minimum_project_options:
    for share_threshold in share_threshold_options:
        selected = supplier_relationships.loc[
            supplier_relationships['supplier_project_count']
            .ge(minimum_projects)
            & supplier_relationships[
                'project_share_within_subagency_pct'
            ].ge(share_threshold)
            & supplier_relationships[
                'value_share_within_subagency_pct'
            ].ge(share_threshold)
        ]

        concentration_sensitivity_records.append({
            'minimum_projects': minimum_projects,
            'share_threshold': share_threshold,
            'relationship_count': len(selected),
            'subagency_count': selected[subagency_column].nunique(),
            'supplier_count': selected[supplier_id_column].nunique()
        })

concentration_sensitivity = pd.DataFrame(
    concentration_sensitivity_records
)

display(concentration_sensitivity)


In [ ]:
high_dependence_relationships = (
    supplier_relationships.loc[
        supplier_relationships['supplier_project_count']
        .ge(minimum_supplier_projects)
        & supplier_relationships[
            'project_share_within_subagency_pct'
        ].ge(minimum_supplier_share)
        & supplier_relationships[
            'value_share_within_subagency_pct'
        ].ge(minimum_supplier_share)
    ]
    .copy()
)

high_dependence_keys = high_dependence_relationships[
    [subagency_column, supplier_id_column]
]

high_dependence_project_ids = (
    study_supplier_entity_data
    .merge(
        high_dependence_keys,
        on=[subagency_column, supplier_id_column],
        how='inner',
        validate='many_to_many'
    )[project_id_column]
    .unique()
)

project_data['flag_high_supplier_dependence'] = (
    project_data['in_study_scope']
    & project_data[project_id_column]
    .isin(high_dependence_project_ids)
)

dependence_summary = pd.Series({
    'ความสัมพันธ์ที่ผ่านเกณฑ์': len(high_dependence_relationships),
    'หน่วยงานย่อยที่เกี่ยวข้อง': (
        high_dependence_relationships[subagency_column].nunique()
    ),
    'ผู้รับจ้างที่เกี่ยวข้อง': (
        high_dependence_relationships[supplier_id_column].nunique()
    ),
    'โครงการที่เกี่ยวข้อง': (
        project_data['flag_high_supplier_dependence'].sum()
    )
}, name='value')

display(dependence_summary)


In [ ]:
sensitivity_heatmap = (
    concentration_sensitivity
    .pivot(
        index='minimum_projects',
        columns='share_threshold',
        values='relationship_count'
    )
)

fig, ax = plt.subplots(figsize=(8, 5))

sns.heatmap(
    sensitivity_heatmap,
    annot=True,
    fmt=',.0f',
    cmap='Blues',
    cbar_kws={'label': 'จำนวนความสัมพันธ์'},
    ax=ax
)

ax.add_patch(
    plt.Rectangle(
        (1, 1),
        1,
        1,
        fill=False,
        edgecolor='#D62728',
        linewidth=3
    )
)

ax.set_title('จำนวนความสัมพันธ์เปลี่ยนตามเกณฑ์ที่เลือก')
ax.set_xlabel('ส่วนแบ่งขั้นต่ำทั้งจำนวนและมูลค่า (%)')
ax.set_ylabel('จำนวนโครงการขั้นต่ำที่ผู้รับจ้างได้รับ')

plt.tight_layout()

figure_path = figure_directory / 'fig04_04_supplier_dependence_sensitivity.png'
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'Figure saved: {figure_path}')


### เหตุผลที่เลือก 10 โครงการและ 75%

ขั้นต่ำ 5 โครงการยังมีฐานเล็กและให้ความสัมพันธ์จำนวนมาก ส่วน 20 โครงการอาจเข้มเกินไปสำหรับหน่วยงานขนาดเล็ก ด้านสัดส่วน 50% แสดงเพียงการได้รับงานมากกว่าครึ่ง ขณะที่ 90% ตัดกรณีออกมาก

จึงเลือกอย่างน้อย 10 โครงการและอย่างน้อย 75% ทั้งจำนวนและมูลค่า เพื่อให้มีฐานงานพอสมควรและสะท้อนการพึ่งพาที่เด่นชัด ค่าเหล่านี้เป็น threshold ของการศึกษา ไม่ใช่ข้อพิสูจน์ว่าการแข่งขันถูกจำกัด


In [ ]:
project_data['flag_material_price_difference'] = (
    project_data['flag_material_above_budget']
    | project_data['flag_material_above_reference']
)

project_data['flag_reference_price_issue'] = (
    project_data['reference_price_status']
    .ne('Within expected range')
)

data_quality_flags = [
    'flag_entity_value_mismatch',
    'flag_reference_price_issue'
]

procurement_review_flags = [
    'flag_material_price_difference',
    'flag_repeated_near_500k_pattern',
    'flag_high_supplier_dependence'
]

project_data['data_quality_indicator_count'] = (
    project_data[data_quality_flags]
    .astype(int)
    .sum(axis=1)
)

project_data['procurement_indicator_count'] = (
    project_data[procurement_review_flags]
    .astype(int)
    .sum(axis=1)
)

study_review_data = project_data.loc[
    project_data['in_study_scope']
].copy()

indicator_count_summary = (
    study_review_data['procurement_indicator_count']
    .value_counts()
    .sort_index()
    .rename_axis('indicator_count')
    .reset_index(name='project_count')
)

indicator_count_summary['project_pct'] = (
    indicator_count_summary['project_count']
    / len(study_review_data)
    * 100
)

display(indicator_count_summary)

indicator_labels = {
    'flag_material_price_difference': 'ส่วนต่างราคา',
    'flag_repeated_near_500k_pattern': 'รูปแบบใกล้เพดานที่เกิดซ้ำ',
    'flag_high_supplier_dependence': 'พึ่งพาผู้รับจ้างสูง'
}

indicator_matrix = (
    study_review_data[procurement_review_flags]
    .astype(bool)
    .rename(columns=indicator_labels)
)

indicator_overlap = indicator_matrix.astype(int).T.dot(
    indicator_matrix.astype(int)
)

display(indicator_overlap)


## 4. จากตัวชี้วัดเดี่ยวสู่รูปแบบซ้อนทับ

การผ่านตัวชี้วัดเดียวอาจมีคำอธิบายปกติได้หลายแบบ จึงใช้การยืนยันข้ามมิติเป็นหลักจัดลำดับ:

- 0 ตัวชี้วัด: ไม่พบสัญญาณตามกติกาที่ใช้
- 1 ตัวชี้วัด: ตรวจสอบตามปกติ
- อย่างน้อย 2 ตัวชี้วัด: ตรวจสอบลำดับแรก

ต้องอ่าน pairwise overlap ควบคู่กับจำนวนรวม เพื่อเห็นว่ารูปแบบซ้อนทับเกิดจากตัวชี้วัดคู่ใด “ตรวจสอบลำดับแรก” ไม่ได้หมายความว่าเป็นโครงการทุจริต


In [ ]:
indicator_counts = (
    indicator_matrix
    .sum()
    .sort_values(ascending=True)
)

combination_summary = (
    indicator_matrix
    .value_counts()
    .reset_index(name='project_count')
)

combination_summary['combination'] = (
    combination_summary
    .apply(
        lambda row: ' + '.join([
            column
            for column in indicator_matrix.columns
            if row[column]
        ]) or 'ไม่พบตัวชี้วัด',
        axis=1
    )
)

flagged_combinations = (
    combination_summary.loc[
        combination_summary['combination'].ne('ไม่พบตัวชี้วัด')
    ]
    .sort_values('project_count', ascending=True)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

bars = axes[0].barh(
    indicator_counts.index,
    indicator_counts.values,
    color='#4C78A8'
)
axes[0].bar_label(bars, fmt='{:,.0f}', padding=4)
axes[0].set_title('จำนวนโครงการที่ผ่านแต่ละตัวชี้วัด')
axes[0].set_xlabel('จำนวนโครงการ')
axes[0].set_ylabel('')
axes[0].spines[['top', 'right']].set_visible(False)

bars = axes[1].barh(
    flagged_combinations['combination'],
    flagged_combinations['project_count'],
    color='#E67E22'
)
axes[1].bar_label(bars, fmt='{:,.0f}', padding=4)
axes[1].set_title('รูปแบบการซ้อนทับของตัวชี้วัด')
axes[1].set_xlabel('จำนวนโครงการ')
axes[1].set_ylabel('')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()

figure_path = figure_directory / 'fig04_05_indicator_counts_and_overlap.png'
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'บันทึกรูป: {figure_path}')


In [ ]:
project_data['review_priority'] = np.select(
    [
        ~project_data['in_study_scope'],
        project_data['procurement_indicator_count'].ge(2),
        project_data['procurement_indicator_count'].eq(1)
    ],
    [
        'นอกขอบเขตศึกษา',
        'ตรวจสอบลำดับแรก',
        'ตรวจสอบตามปกติ'
    ],
    default='ไม่พบตัวชี้วัด'
)

study_review_data = project_data.loc[
    project_data['in_study_scope']
].copy()

review_priority_summary = (
    study_review_data
    .groupby('review_priority')
    .agg(
        project_count=(project_id_column, 'count'),
        total_awarded_value=(awarded_price_column, 'sum')
    )
    .reset_index()
)

review_priority_summary['project_pct'] = (
    review_priority_summary['project_count']
    / len(study_review_data)
    * 100
)

review_priority_summary['awarded_value_million'] = (
    review_priority_summary['total_awarded_value']
    / 1_000_000
)

display(review_priority_summary)


In [ ]:
study_base_count = len(study_review_data)

stage_counts = [
    study_base_count,
    (
        study_review_data['procurement_indicator_count']
        .ge(1).sum()
    ),
    (
        study_review_data['procurement_indicator_count']
        .ge(2).sum()
    )
]

stage_labels = [
    'กลุ่มศึกษาหลัก',
    'พบอย่างน้อย 1 ตัวชี้วัด',
    'ตรวจสอบลำดับแรก'
]

stage_pct = [
    count / study_base_count * 100
    for count in stage_counts
]

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.axis('off')

x_positions = [0.12, 0.50, 0.88]

for index, (x_position, label, count, percentage) in enumerate(
    zip(x_positions, stage_labels, stage_counts, stage_pct)
):
    color = '#B43C2F' if index == 2 else (
        '#E67E22' if index == 1 else '#4C78A8'
    )

    ax.text(
        x_position,
        0.58,
        f'{count:,.0f}',
        ha='center',
        va='center',
        fontsize=20,
        fontweight='bold',
        color='white',
        bbox={
            'boxstyle': 'round,pad=0.8',
            'facecolor': color,
            'edgecolor': 'none'
        }
    )
    ax.text(
        x_position,
        0.27,
        f'{label}\n({percentage:.2f}% ของกลุ่มศึกษา)',
        ha='center',
        va='center',
        fontsize=11
    )

    if index < len(x_positions) - 1:
        ax.annotate(
            '',
            xy=(x_positions[index + 1] - 0.11, 0.58),
            xytext=(x_position + 0.11, 0.58),
            arrowprops={
                'arrowstyle': '->',
                'color': '#7F7F7F',
                'linewidth': 1.5
            }
        )

ax.set_title(
    'การลดขอบเขตจากกลุ่มศึกษาหลักสู่โครงการตรวจสอบลำดับแรก',
    fontsize=16
)

plt.tight_layout()

figure_path = (
    figure_directory
    / 'fig04_06_review_prioritization.png'
)
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'Figure saved: {figure_path}')


In [ ]:
high_priority_columns = [
    project_id_column,
    project_name_column,
    agency_column,
    subagency_column,
    supplier_name_column,
    method_column,
    budget_column,
    reference_price_column,
    awarded_price_column,
    'flag_material_above_budget',
    'flag_material_above_reference',
    'flag_repeated_near_500k_pattern',
    'flag_high_supplier_dependence',
    'procurement_indicator_count',
    'data_quality_indicator_count'
]

high_priority_cases = (
    project_data.loc[
        project_data['review_priority'].eq('ตรวจสอบลำดับแรก'),
        high_priority_columns
    ]
    .sort_values(
        ['procurement_indicator_count', awarded_price_column],
        ascending=False
    )
)

display(high_priority_cases.head(20))


In [ ]:
concentration_case = (
    high_dependence_relationships
    .sort_values(
        [
            'project_share_within_subagency_pct',
            'value_share_within_subagency_pct',
            'supplier_project_count'
        ],
        ascending=False
    )
    .head(1)
)

display(concentration_case)

if not concentration_case.empty:
    concentration_case_projects = (
        project_data.loc[
            project_data[subagency_column].eq(
                concentration_case.iloc[0][subagency_column]
            )
            & project_data[supplier_id_column].eq(
                concentration_case.iloc[0][supplier_id_column]
            )
        ]
    )

    display(
        concentration_case_projects[
            [
                project_id_column,
                project_name_column,
                budget_column,
                awarded_price_column,
                transaction_date_column
            ]
        ].head(20)
    )


In [ ]:
# Inspect contract rows of the unique price-and-pattern case
case_project_id = 69049225316

case_contract_data = contract_data.loc[
    contract_data[project_id_column].eq(case_project_id)
].copy()

case_contract_columns = [
    project_id_column,
    project_name_column,
    supplier_id_column,
    supplier_name_column,
    contract_number_column,
    contract_budget_column,
    awarded_price_column,
    'วันที่ลงนามในสัญญา',
    'วันที่สิ้นสุดสัญญา',
    'is_joint_venture_member',
    'source_file'
]

display(
    case_contract_data[
        case_contract_columns
    ].style.format({
        contract_budget_column: '{:,.2f}',
        awarded_price_column: '{:,.2f}'
    })
)

case_contract_summary = pd.DataFrame({
    'value': [
        len(case_contract_data),
        case_contract_data[supplier_id_column].nunique(),
        case_contract_data[contract_number_column].nunique(),
        case_contract_data[contract_budget_column].sum(),
        case_contract_data[awarded_price_column].iloc[0]
    ]
}, index=[
    'Contract-level rows',
    'Unique supplier IDs',
    'Unique contract numbers',
    'Sum of contract values',
    'Project awarded total'
])

display(case_contract_summary)

## 5. กรณีศึกษา

### กรณี A: ส่วนต่างราคาและหลายสัญญา

โครงการ `69049225316` มีงบประมาณและราคากลาง 491,000 บาท แต่ราคาที่ตกลงรวม 980,000 บาท ข้อมูลสัญญาพบ 2 ฉบับกับผู้รับจ้างรายเดียว ฉบับละ 490,000 บาท

ยอดสัญญารวมตรงกับยอดระดับโครงการ สิ่งที่ต้องตรวจต่อคือขอบเขตของแต่ละสัญญา แหล่งงบ การอนุมัติเพิ่มหรือแก้ไข และลำดับเหตุการณ์


### กรณี B: รูปแบบใกล้เส้นแบ่งที่เกิดซ้ำ

กลุ่มใหญ่ที่สุดเป็น 65 โครงการของสำนักงานทรัพยากรธรรมชาติและสิ่งแวดล้อมจังหวัดมุกดาหาร ผู้รับจ้างคือบริษัท บ้านบุ่ง วิศวกรรม จำกัด และมีวันที่เกิดรายการ 29 ธันวาคม 2568

ควรตรวจความคล้ายของงาน สถานที่ แผนจัดซื้อ TOR/BOQ และแหล่งงบ ก่อนพิจารณาว่าโครงการมีเหตุผลให้ดำเนินการแยกกันหรือไม่

### กรณี C: ความสัมพันธ์ที่พึ่งพาผู้รับจ้างสูง

เทศบาลตำบลบ้านโปร่งมี 73 โครงการในกลุ่มศึกษา และผู้รับจ้างรายเดียวได้รับครบทั้ง 73 โครงการ คิดเป็น 100% ทั้งจำนวนและมูลค่า รวมประมาณ 7.97 ล้านบาท

ข้อมูลผู้ชนะอย่างเดียวยังอธิบายไม่ได้ว่าเกิดจากการแข่งขันต่ำ ความเชี่ยวชาญ หรือข้อจำกัดด้านพื้นที่ จึงต้องตรวจจำนวนผู้เสนอราคา วิธีเชิญผู้ประกอบการ และความสัมพันธ์ของขอบเขตงาน

### กรณี D: การซ้อนทับหลายมิติ

โครงการตรวจสอบลำดับแรกมี 296 โครงการ โดย 291 โครงการมาจากรูปแบบใกล้เพดานที่เกิดซ้ำร่วมกับการพึ่งพาผู้รับจ้างสูง อีก 5 โครงการเกี่ยวข้องกับส่วนต่างราคา

การนำเสนอควรเลือกอย่างน้อยหนึ่งโครงการจากแต่ละรูปแบบการซ้อนทับ เพื่อครอบคลุมราคา รูปแบบเกิดซ้ำ และการพึ่งพาผู้รับจ้าง


## 6. สรุปผล

| ผลการคัดกรองในกลุ่มศึกษา | จำนวนโครงการ |
|---|---:|
| ส่วนต่างราคา | 299 |
| รูปแบบใกล้เพดานที่เกิดซ้ำ | 5,623 |
| ความสัมพันธ์ที่พึ่งพาผู้รับจ้างสูง | 2,421 |
| พบอย่างน้อย 1 ตัวชี้วัด | 8,047 |
| ตรวจสอบลำดับแรก: พบ 2 ตัวชี้วัด | 296 |

Insight หลักคือการอยู่ใกล้ 500,000 บาทเพียงอย่างเดียวไม่เพียงพอ กลุ่มตรวจสอบลำดับแรกส่วนใหญ่เกิดจากการซ้อนทับระหว่างรูปแบบใกล้เพดานที่เกิดซ้ำกับการพึ่งพาผู้รับจ้างรายเดียวสูงภายในหน่วยงานย่อย

ผลลัพธ์ช่วยลดขอบเขตการเปิดเอกสาร แต่ยังไม่อธิบายสาเหตุหรือพิสูจน์ความผิด


## 7. ข้อจำกัดและการนำผลไปใช้

- ข้อมูลสะสมถึงวันที่ 30 กรกฎาคม 2569 ไม่ใช่ข้อมูลเต็มปี
- มีเฉพาะผู้ชนะ ไม่มีผู้เสนอราคาและราคาที่เสนอทั้งหมด
- วันที่เกิดรายการอาจไม่ใช่วันที่ตัดสินใจจัดซื้อหรือวันที่ลงนาม
- ช่วง 490,000–500,000 บาท, 3 โครงการ, 10,000 บาท, 1%, 10 โครงการ และ 75% เป็นเกณฑ์ของการศึกษา
- ผลการทดสอบความไวแสดงว่าจำนวนที่ถูกคัดกรองเปลี่ยนตาม threshold
- การอยู่ใกล้เส้นแบ่ง การเกิดซ้ำ และการพึ่งพาผู้รับจ้างไม่พิสูจน์เจตนาหรือความผิด
- ต้องตรวจแผนจัดซื้อ TOR/BOQ แหล่งงบ เอกสารอนุมัติ สัญญา และข้อมูลผู้เสนอราคาก่อนสรุปสาเหตุ


In [ ]:
# Export review outputs for later reporting
output_directory = processed_dir
output_directory.mkdir(parents=True, exist_ok=True)

project_review_columns = [
    project_id_column,
    project_name_column,
    agency_column,
    subagency_column,
    province_column,
    method_column,
    supplier_id_column,
    supplier_name_column,
    transaction_date_column,
    budget_column,
    reference_price_column,
    awarded_price_column,
    'in_study_scope',
    'reference_price_status',
    'flag_reference_price_issue',
    'flag_entity_value_mismatch',
    'flag_multi_party_contract',
    'budget_overrun',
    'budget_overrun_pct',
    'reference_overrun',
    'reference_overrun_pct',
    'near_500k_cluster_size',
    'flag_material_price_difference',
    'flag_repeated_near_500k_pattern',
    'flag_high_supplier_dependence',
    'data_quality_indicator_count',
    'procurement_indicator_count',
    'review_priority'
]

project_review_output = project_data[
    project_review_columns
].copy()

study_review_output = project_review_output.loc[
    project_review_output['in_study_scope']
].copy()

high_priority_output = project_review_output.loc[
    project_review_output['review_priority']
    .eq('ตรวจสอบลำดับแรก')
].copy()

repeated_cluster_output = (
    near_500k_clusters.loc[
        near_500k_clusters['project_count']
        .ge(minimum_cluster_size)
    ]
    .sort_values(
        ['project_count', 'total_budget'],
        ascending=False
    )
)

output_objects = {
    'project_review_indicators_2569.csv': project_review_output,
    'study_scope_review_indicators_2569.csv': study_review_output,
    'high_priority_projects_2569.csv': high_priority_output,
    'repeated_near_500k_clusters_2569.csv': repeated_cluster_output,
    'high_supplier_dependence_2569.csv': high_dependence_relationships,
    'price_threshold_sensitivity_2569.csv': materiality_sensitivity,
    'near_500k_sensitivity_2569.csv': near_ceiling_sensitivity,
    'cluster_size_sensitivity_2569.csv': cluster_size_sensitivity,
    'supplier_dependence_sensitivity_2569.csv': concentration_sensitivity
}

export_records = []

for file_name, output_data in output_objects.items():
    output_path = output_directory / file_name
    output_data.to_csv(
        output_path,
        index=False,
        encoding='utf-8-sig'
    )
    export_records.append({
        'file_name': file_name,
        'rows': len(output_data),
        'file_size_mb': output_path.stat().st_size / 1024**2
    })

export_summary = pd.DataFrame(export_records)
display(export_summary)


## 8. ไฟล์ผลลัพธ์

Notebook ส่งออกข้อมูลระดับโครงการ กลุ่มศึกษาหลัก โครงการตรวจสอบลำดับแรก กลุ่มเกิดซ้ำ ความสัมพันธ์ที่พึ่งพาผู้รับจ้างสูง และผล sensitivity ของ:

- หน้าต่างใกล้เพดาน
- จำนวนขั้นต่ำของกลุ่มเกิดซ้ำ
- ส่วนต่างราคา
- การพึ่งพาผู้รับจ้าง

ผลลัพธ์เป็นการคัดกรองจากข้อมูล ไม่ใช่หลักฐานหรือข้อสรุปเกี่ยวกับการทุจริต
